# Gün 140: TCP Reliability, Retransmission ve Ordering (Güvenilirlik, Yeniden İletim, Sıralama, Jacobson/Karels RTO ve Fast Retransmit)

```
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR

Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)

Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim
amaçlı olarak paylaşılmıştır.

YASAKLAR:
  1. Kopyalanamaz, çoğaltılamaz, dağıtılamaz veya yeniden yayınlanamaz.
  2. Ticari veya ticari olmayan hiçbir projede kullanılamaz, değiştirilemez.
  3. Alt lisanslanamaz, satılamaz veya devredilemez.
  4. Tersine mühendislik yapılamaz.

İZİN VERİLEN KULLANIM:
  - GitHub üzerinde görüntüleme ve okuma.
  - Kişisel öğrenim amacıyla kodu inceleme (kopyalamadan).

YAZARIN AÇIK YAZILI İZNİ OLMAKSIZIN HİÇBİR KULLANIM HAKKI TANINMAZ.
İzin talepleri için: GitHub @seydivakkas
```

[![License: All Rights Reserved](https://img.shields.io/badge/license-All%20Rights%20Reserved-red?style=flat-square)](https://github.com/seydivakkas)

**Faz:** F4 — TCP/UDP / Protocol / Serialization  
**Konu:** L4 Taşıma Katmanı Güvenilirlik Mekanizmaları: İletim Kontrol Protokolü (TCP - RFC 793 / RFC 6298 / RFC 5681 / RFC 2018), Kümülatif Onay (Cumulative ACK), Seçici Onay (SACK - Selective Acknowledgment), Jacobson/Karels RTT/RTO Dinamik Zaman Aşımı Tahmini, Karn Algoritması ve Üstel Geri Çekilme (Exponential Backoff), Hızlı Yeniden İletim (Fast Retransmit & 3 DUP ACK), Alıcı Sıralama Tamponu (Out-of-Order Reordering Buffer)  
**Ön Koşul:** Gün 139 — TCP Handshake ve Connection Lifecycle  
**Kritik Seviye:** Savunma Sanayii Uydu, Telsiz ve İHA/SİHA Taktik Veri Bağlarında (Data Link) Paket Kaybı, Sıra Bozulması ve Gecikme Dalgalanmasında (Jitter) Kesintisiz Sıralı Telemetri İletimi

---

### 40-Nokta Bildirimi ve Kalite Güvencesi
Bu modül; IP katmanının güvenilmez, paket kaybedebilen ve sıra bozabilen doğası üzerinde TCP'nin nasıl **%100 kayıpsız, tekilleştirilmiş ve kusursuz sıralı (Strict In-Order)** bir veri boru hattı inşa ettiğini derinlemesine inceler. Bayt bazlı kümülatif ACK mekanizmasının sınırları, ağ gecikmesine dinamik uyum sağlayan **Jacobson/Karels RTT/RTO algoritması** ($SRTT, RTTVAR, RTO$), RTO bekleme cezasını ortadan kaldıran **Hızlı Yeniden İletim (3 DUP ACK Fast Retransmit)** ve alıcı çekirdeğindeki `tcp_ofo_queue` (Out-of-Order Red-Black Tree) sıralama tamponu çalışan C++20 mimarili simülatörle kanıtlanmıştır.


## 1. 🎯 Bugünkü Problem: Taktik Sahada Paketler Kaybolur, Karışır ve Gecikir!

Askeri bir İHA/SİHA yer kontrol istasyonuna uydu veya taktik RF telsiz hattı üzerinden radar hedef izleme telemetrisi akıtmaktadır. 

Taktik haberleşme ortamının kaçınılmaz fiziksel gerçekleri:
1. **Paket Kaybı (Packet Loss):** Elektronik harp karıştırması (jamming) veya hava koşulları nedeniyle paketlerin %5'i havada yok olur (Drop).
2. **Sıra Bozulması (Packet Reordering / Out-of-Order):** Yönlendiriciler (Routers) paketleri çok yollu (multi-path) rotalardan geçirir; 4. paket 2. paketten önce üsse ulaşır.
3. **Gecikme Dalgalanması (Jitter):** Uydu yörünge hareketi veya tampon birikmesi (Bufferbloat) nedeniyle RTT $20\text{ ms}$ ile $400\text{ ms}$ arasında vahşice dalgalanır.

Eğer alıcı yazılım UDP gibi çalışsaydı:
- Hedef koordinatları (`X=120`, `Y=45`) yerine sırası karışmış (`Y=45`, `X=120`) baytları parse eder ve hava savunma füzesi dost unsuru vurabilirdi!
- Kaybolan paketler telafi edilemez, radar ekranında hedef birden yok olurdu.

Peki TCP bunu nasıl çözer?
- Sabit bir zaman aşımı koyamazsınız: $100\text{ ms}$ koysanız fiberde çok yavaş, uyduda ise sürekli yanlış yere paket tekrarı (Spurious Retransmit) yapar!
- Kayıp paketi anlamak için her defasında 1 saniye bekleyemezsiniz; sistem donar (Head-of-Line Blocking)!

Bugün; TCP'nin ağın nabzını tutan matematiksel RTO algoritmasını, 3 adet yinelenen ACK ile anında kayıp tespit eden Fast Retransmit mekanizmasını ve delikli alıcı tamponunu (Reordering Buffer) bizzat kodlayarak çözeceğiz!


## 2. 🧠 Temel Teori: Kümülatif ACK, SACK ve Out-of-Order Sıralama

TCP'de sıra numaraları paket bazlı değil, **bayt bazlıdır**. İletilen her veri baytına 32-bitlik bir sıra numarası (`Seq`) atanır.

### Kümülatif Onay (Cumulative ACK) Mantığı
Alıcının yolladığı `ACK = N` şu kesin taahhüttür:
> *"Sıra numarası $N-1$'e kadar olan TÜM baytları eksiksiz, ardışık ve hasarsız aldım. Sırada beklediğim ilk bayt $N$'dir."*

```
Gönderici:  [Paket 1: 100-199]  [Paket 2: 200-299 (KAYIP!)]  [Paket 3: 300-399]  [Paket 4: 400-499]
                    |                     X                       |                   |
Alıcı:        ACK = 200                                       ACK = 200           ACK = 200
                                                            (1. DUP ACK)        (2. DUP ACK)
```

Alıcı Paket 3 ve Paket 4'ü alsa bile `ACK` numarasını 300 veya 500 yapamaz! Çünkü 200-299 arasındaki baytlar henüz eline geçmemiştir. Alıcı sadece beklediği baytı (`ACK = 200`) tekrar tekrar bildirir.

---

### Seçici Onay (SACK - Selective Acknowledgment, RFC 2018)
Kümülatif ACK'in körlüğünü aşmak için geliştirilmiştir. TCP başlığının opsiyon alanında (TCP Options) birincil kümülatif ACK'e ek olarak **ulaşmış olan izole bayt aralıkları (SACK Blokları)** raporlanır:
```text
ACK = 200, SLE=300, SRE=500 (300 ile 500 arasındaki baytlar elimde!)
```
Bu sayede gönderici Paket 3 ve 4'ün ulaştığını kesin olarak anlar ve gereksiz yere sadece ve sadece **Paket 2'yi (200-299)** yeniden iletir.

---

### Alıcı Sıralama Tamponu (Out-of-Order Reordering Buffer)
Alıcıya sırasız (out-of-order) gelen segmentler çöpe atılmaz. Linux çekirdeğinde `struct tcp_sock` içinde `tcp_ofo_queue` adında bir **Kırmızı-Siyah Ağaçta (Red-Black Tree)** sıralı olarak bekletilir.
1. `rcv_nxt` (beklenen bayt) gelmediği sürece bu veriler kullanıcıya (`read()`) verilmez.
2. Kayıp segment nihayet geldiğinde, tamponda hazır bekleyen ardışık bloklar tek seferde `receive_queue`'ya kaydırılır ve tek bir büyük `ACK` fırlatılır.


## 3. 🔬 Derinlemesine Mekanizma: Jacobson/Karels Dinamik RTT/RTO ve Fast Retransmit

Statik bir zaman aşımı (timeout) internet ve taktik ağlarda felakettir. TCP, her iletilen paketin gidiş-dönüş süresini ($RTT$) sürekli örnekleyerek **Retransmission Timeout (RTO)** süresini dinamik olarak hesaplar (RFC 6298).

### Jacobson / Karels Algoritması (RFC 6298)

1. **Örnek RTT ($R$):** Bir paketin gönderilmesi ile ACK'inin alınması arasındaki süre.
2. **Düzeltilmiş RTT (Smoothed RTT - $SRTT$):** Üstel ağırlıklı hareketli ortalama (EWMA):
   $$SRTT \leftarrow (1 - \alpha) \cdot SRTT + \alpha \cdot R \quad (\alpha = 1/8 = 0.125)$$
3. **RTT Varyansı (RTT Variation - $RTTVAR$):** Gecikme dalgalanması / Jitter tahmini:
   $$RTTVAR \leftarrow (1 - \beta) \cdot RTTVAR + \beta \cdot |SRTT - R| \quad (\beta = 1/4 = 0.25)$$
4. **Zaman Aşımı Süresi ($RTO$):**
   $$RTO \leftarrow SRTT + \max(G, 4 \cdot RTTVAR)$$
   - Burada $G$: Sistem saat çözünürlüğü (Clock Granularity, örn. $1\text{ ms}$).
   - Linux çekirdeğinde minimum limit: $RTO_{\min} = 200\text{ ms}$ (RFC standardı $1000\text{ ms}$), maksimum limit: $RTO_{\max} = 120\text{ saniye}$.

```
                  RTT Örnekleri (R)
                         |
                         v
             +-----------------------+
             |      SRTT (EWMA)      | <--- alpha = 1/8
             +-----------------------+
                         |
                         v
             +-----------------------+
             |    RTTVAR (Sapma)     | <--- beta = 1/4
             +-----------------------+
                         |
                         v
             +-----------------------+
             | RTO = SRTT + 4*RTTVAR | ---> [Zaman Aşımı Sayacı]
             +-----------------------+
```

---

### Karn Algoritması ve Üstel Geri Çekilme (Exponential Backoff)
1. **Yeniden İletim Belirsizliği (Retransmission Ambiguity):** Eğer bir paket kaybolup tekrar gönderildiyse, gelen ACK ilk pakete mi yoksa ikinci pakete mi aittir? Bilinemez! Bu yüzden **yeniden iletilen paketlerden RTT örneği ALINMAZ**.
2. **Exponential Backoff:** RTO süresi dolup paket zaman aşımına uğradığında, ağda ciddi bir tıkanıklık (congestion collapse) olduğu varsayılır ve RTO her denemede ikiye katlanır:
   $$RTO \leftarrow 2 \times RTO$$

---

### Hızlı Yeniden İletim (Fast Retransmit) ve 3 DUP ACK Kuralı (RFC 5681)
RTO zaman aşımı çok yavaştır (en az $200\text{ ms}$ veya $1000\text{ ms}$). Gerçek zamanlı bir sistem bu kadar bekleyemez.

**Kurtarıcı Kural:**
- Alıcı kayıp bir paketin ardından gelen her out-of-order segment için göndericiye **aynı ACK numarasını içeren yinelenen bir onay (Duplicate ACK)** gönderir.
- Gönderici **3 ADET YİNELENEN ACK (3 DUP ACK)** aldığı anda:
  1. Paketin sadece yolda biraz gecikmediğini, kesinlikle düştüğünü anlar.
  2. RTO zaman aşımı sayacının dolmasını beklemeden kayıp segmenti **ANINDA TEKRAR İLETİR (Fast Retransmit)!**
  3. Gecikme $1000\text{ ms}$'den **$1\text{ RTT}$ ($10 - 30\text{ ms}$)** seviyesine iner!


## 4. ⚙️ Kritik Linux Çekirdek Parametreleri ve RACK Mimarisi

### Linux TCP Güvenilirlik Parametreleri (`/proc/sys/net/ipv4/`)

| Parametre | Varsayılan | Savunma Sanayii / Kritik Öneri | Açıklama |
|:---|:---|:---|:---|
| `tcp_sack` | 1 | **1 (Zorunlu)** | RFC 2018 Seçici Onaylama (SACK). Kesinlikle kapatılmamalıdır. |
| `tcp_dsack` | 1 | **1 (Zorunlu)** | Yinelenen SACK (Duplicate SACK). Sahte yeniden iletimleri tespit eder. |
| `tcp_reordering` | 3 | **3 - 10** | Fast Retransmit'i tetikleyen DUP ACK eşiği (Varsayılan 3). |
| `tcp_retries1` | 3 | 3 | IP katmanı rota sorgulaması öncesi yeniden deneme sayısı. |
| `tcp_retries2` | 15 | 5 - 8 | Bağlantıyı öldürmeden önce toplam yeniden iletim denemesi (~15 dk yerine ~3 dk). |
| `tcp_recovery` | 1 (RACK) | **1 (RACK Aktif)** | Modern Zaman Tabanlı Kayıp Tespiti (Recent ACKs - RFC 8985). |

---

### Modern Devrim: RACK (Recent ACKs - RFC 8985)
Geleneksel 3 DUP ACK kuralı, gönderilecek az sayıda paket kaldığında (örn. pencerenin sonunda 1 veya 2 paket varsa) 3 adet DUP ACK üretemez ve RTO zaman aşımına düşer.

Linux çekirdeğinin modern standardı olan **RACK**, paket sıra numaraları yerine **zaman damgalarını (TCP Timestamps - RFC 7323)** izler. Daha geç gönderilen bir paketin ACK'i alındığında, daha önce gönderilen paket henüz onaylanmamışsa ve aradan $1\times RTT + \text{ReoWnd}$ kadar süre geçmişse, o paket 3 DUP ACK beklenmeksizin kayıp ilan edilir!


## 5. 💻 1. Kod Bloğu: C++20 / Python TCP Reliability ve RTO Hesaplama Motoru

Aşağıdaki kod; RFC 6298 uyumlu **Jacobson/Karels RTT/RTO Hesaplayıcısını**, bayt seviyesinde kümülatif ve seçici (SACK) onay yöneticisini ve alıcı tarafındaki **Out-of-Order Sıralama Tamponunu (`TCPReorderingBuffer`)** simüle eden çekirdek motoru tanımlar.


In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 140: Kod Bloğu 1 — TCP Reliability, RTO ve Out-of-Order Sıralama Motoru
Jacobson/Karels RTT/RTO Algoritması (RFC 6298) ve Reordering Buffer.
"""

import os
import sys
import time
import math
import random
from typing import Dict, List, Tuple, Optional, NamedTuple

class RTTStats:
    """RFC 6298 Jacobson/Karels Algoritması Gerçeklemesi"""
    def __init__(self, alpha: float = 0.125, beta: float = 0.25, g_sec: float = 0.001):
        self.alpha = alpha
        self.beta = beta
        self.g = g_sec  # Clock granularity (1 ms)
        self.srtt: Optional[float] = None
        self.rttvar: Optional[float] = None
        self.rto: float = 1.0  # RFC başlangıç RTO = 1.0 saniye
        self.rtt_history: List[Tuple[float, float, float, float]] = [] # (R, SRTT, RTTVAR, RTO)

    def update(self, r_sample: float):
        """Yeni bir RTT örneği ile SRTT, RTTVAR ve RTO güncellenir"""
        if self.srtt is None:
            # İlk ölçüm (RFC 6298 Bölüm 2.2)
            self.srtt = r_sample
            self.rttvar = r_sample / 2.0
            self.rto = self.srtt + max(self.g, 4.0 * self.rttvar)
        else:
            # Sonraki ölçümler (RFC 6298 Bölüm 2.3)
            self.rttvar = (1.0 - self.beta) * self.rttvar + self.beta * abs(self.srtt - r_sample)
            self.srtt = (1.0 - self.alpha) * self.srtt + self.alpha * r_sample
            self.rto = self.srtt + max(self.g, 4.0 * self.rttvar)

        # Sınırlandırmalar: min 0.2s (Linux standardı), max 60s
        self.rto = max(0.2, min(self.rto, 60.0))
        self.rtt_history.append((r_sample, self.srtt, self.rttvar, self.rto))

    def backoff(self):
        """Karn Exponential Backoff: RTO ikiye katlanır"""
        self.rto = min(self.rto * 2.0, 60.0)

class TCPSegment(NamedTuple):
    seq: int
    data: bytes
    send_time: float
    is_retransmit: bool = False

class TCPReorderingBuffer:
    """Alıcı Çekirdek Sıralama Tamponu (Out-of-Order Queue / Red-Black Tree Mantığı)"""
    def __init__(self, initial_expected_seq: int = 1000):
        self.rcv_nxt = initial_expected_seq  # Alıcının beklediği bir sonraki bayt
        self.ofo_buffer: Dict[int, bytes] = {} # seq -> data (Sırasız gelen paketler)
        self.delivered_stream: bytearray = bytearray()
        self.dup_ack_count = 0
        self.last_ack_sent = initial_expected_seq

    def receive_segment(self, segment: TCPSegment) -> Tuple[int, List[Tuple[int, int]]]:
        """
        Gelen segmenti işler.
        Dönüş: (Kümülatif ACK, SACK Blokları [(start, end)])
        """
        seq = segment.seq
        length = len(segment.data)

        # 1. Beklenen segment tam vaktinde geldi mi?
        if seq == self.rcv_nxt:
            self.delivered_stream.extend(segment.data)
            self.rcv_nxt += length
            self.dup_ack_count = 0

            # Şimdi tamponda bekleyen sıradaki paketlerin önü açıldı mı? (Delik kapandı!)
            while self.rcv_nxt in self.ofo_buffer:
                contiguous_data = self.ofo_buffer.pop(self.rcv_nxt)
                self.delivered_stream.extend(contiguous_data)
                self.rcv_nxt += len(contiguous_data)

        elif seq > self.rcv_nxt:
            # 2. Gelecekten bir paket geldi (Out-of-Order)! Tampona al, delik oluştu!
            if seq not in self.ofo_buffer:
                self.ofo_buffer[seq] = segment.data
            self.dup_ack_count += 1
        else:
            # 3. Zaten alınmış eski/yinelenen paket (Duplicate)
            pass

        self.last_ack_sent = self.rcv_nxt
        sack_blocks = self._generate_sack_blocks()
        return self.rcv_nxt, sack_blocks

    def _generate_sack_blocks(self) -> List[Tuple[int, int]]:
        """Tampondaki izole blokları SACK formatında listeler"""
        blocks = []
        for seq in sorted(self.ofo_buffer.keys()):
            blocks.append((seq, seq + len(self.ofo_buffer[seq])))
        return blocks

print("[HÜCRE 7] RTTStats, TCPSegment ve TCPReorderingBuffer başarıyla yüklendi.")


## 6. 💻 2. Kod Bloğu: Sırasız İletim (Out-of-Order) ve Delik Kapatma Deneyi

Bu deneyde:
1. Gönderici 5 adet füze telemetri segmenti gönderir (`Seq=1000`'den başlayarak her biri 100 bayt).
2. Ağda Paket 2 yolda gecikir veya rotası uzar; alıcıya **1, 3, 4, 5, 2** sırasında ulaşır.
3. Alıcının her adımda döndüğü `ACK` ve `SACK` blokları izlenir.
4. Geciken Paket 2 ulaştığı an delik kapanır ve alıcı tek bir dev kümülatif `ACK = 1500` fırlatır!


In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 140: Kod Bloğu 2 — Out-of-Order İletim ve Reordering Buffer Delik Kapatma Deneyi
"""

print("")
print("[*] Out-of-Order Paket İletimi ve Sıralama Tamponu Deneyi Başlatılıyor...")

# 500 Baytlık Taktik Görev Emri
payloads = [
    b"CMD_TARGET_LOCK_AZIMUTH_045_RADAR_NORTH",     # 1000..1099 (100B yapalım)
    b"CMD_MISSILE_WARHEAD_ARM_SAFE_BATTERY_01",    # 1100..1199
    b"CMD_IFF_CHALLENGE_TRANSPONDER_MODE_5_OK",    # 1200..1299
    b"CMD_ENGAGE_TRACK_ID_9942_AIR_SUPERIOR",      # 1300..1399
    b"CMD_TERMINAL_HOMING_ACTIVE_RADAR_SEEKER"     # 1400..1499
]
# Her birini tam 100 bayta pad edelim
segments = []
for idx, p in enumerate(payloads):
    padded = p.ljust(100, b"#")
    segments.append(TCPSegment(seq=1000 + idx * 100, data=padded, send_time=time.time()))

# Senaryo: Sıra bozuldu! Alıcıya geliş sırası: 0, 2, 3, 4, 1 (Paket 1 gecikti!)
arrival_order = [0, 2, 3, 4, 1]
receiver_buffer = TCPReorderingBuffer(initial_expected_seq=1000)

print(f"  [+] Gönderilen Paketler: 5 Adet x 100 Bayt (Seq: 1000 - 1500)")
print(f"  [+] Ağ Varış Sırası   : {[s.seq for s in [segments[i] for i in arrival_order]]}")

for step, arr_idx in enumerate(arrival_order, start=1):
    seg = segments[arr_idx]
    cum_ack, sack_blocks = receiver_buffer.receive_segment(seg)
    print(""); print(f"  [Adım {step}] Paket Geldi: Seq={seg.seq}..{seg.seq+len(seg.data)}")
    print(f"      -> Alıcı Kümülatif ACK: {cum_ack}")
    print(f"      -> SACK Blokları     : {sack_blocks}")
    print(f"      -> DUP ACK Sayısı    : {receiver_buffer.dup_ack_count}")
    print(f"      -> Tampon Bekleyen   : {list(receiver_buffer.ofo_buffer.keys())}")

# Doğrulama
assert receiver_buffer.rcv_nxt == 1500, "HATA: Tüm paketler geldikten sonra rcv_nxt 1500 olmalıydı!"
assert len(receiver_buffer.ofo_buffer) == 0, "HATA: Tüm delikler kapandıktan sonra tampon boş olmalıydı!"
assert len(receiver_buffer.delivered_stream) == 500, "HATA: Teslim edilen veri 500 bayt olmalıydı!"
print(""); print("  [+] Mükemmel Başarı: Out-of-Order paketler %100 kusursuz sırada teslim edildi!")
print("[HÜCRE 9] Sırasız paket sıralama deneyi başarıyla tamamlandı.")


## 7. 💻 3. Kod Bloğu: Sabotaj Laboratuvarı — Paket Kaybı, RTO vs 3 DUP ACK Fast Retransmit

Bu sabotaj testinde iki farklı kayıp kurtarma mekanizmasını çarpıştıracağız:
1. **Sabotaj 1: RTO Zaman Aşımı Beklemesi (Slow Path):** Paket kaybolduğunda ve arkasından başka paket gelmediğinde (örneğin akışın son paketi), sistem çaresizce RTO süresinin ($1000\text{ ms}$) dolmasını bekler. Üstel geri çekilme (Exponential Backoff) tetiklenir.
2. **Sabotaj 2: 3 DUP ACK ile Hızlı Yeniden İletim (Fast Retransmit):** Paket kaybolur ancak arkasından diğer segmentler akmaya devam eder. 3. DUP ACK geldiği an RTO beklenmeden **$1\text{ RTT}$ ($20\text{ ms}$)** içinde anında yeniden iletilir. Kurtarma tam **$50\text{ kat daha hızlı}$** gerçekleşir!


In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 140: Kod Bloğu 3 — Sabotaj Laboratuvarı: RTO Zaman Aşımı vs Fast Retransmit
"""

print("")
print("[*] TCP Kayıp Kurtarma ve Sabotaj Laboratuvarı Başlatılıyor...")

class TCPSenderSimulation:
    def __init__(self, rtt_ms: float = 20.0):
        self.rtt_ms = rtt_ms
        self.rtt_stats = RTTStats()
        self.rtt_stats.update(rtt_ms / 1000.0) # İlk RTT'yi öğret
        self.retransmission_log: List[Dict] = []

    def simulate_rto_recovery(self, lost_seq: int) -> float:
        """Akışın son paketi kayboldu; DUP ACK gelmiyor, mecburen RTO bekleniyor"""
        rto_duration_ms = self.rtt_stats.rto * 1000.0
        self.rtt_stats.backoff() # Karn backoff
        self.retransmission_log.append({
            "type": "RTO_TIMEOUT",
            "seq": lost_seq,
            "latency_ms": rto_duration_ms
        })
        return rto_duration_ms

    def simulate_fast_retransmit(self, lost_seq: int) -> float:
        """Kayıp paketin ardından 3 DUP ACK geldi; anında Fast Retransmit!"""
        # 3 DUP ACK üretilmesi için geçen süre: ~1 RTT
        # Tekrar iletimin onaylanması: +1 RTT. Toplam: 2 RTT
        recovery_time_ms = 2.0 * self.rtt_ms
        self.retransmission_log.append({
            "type": "FAST_RETRANSMIT",
            "seq": lost_seq,
            "latency_ms": recovery_time_ms
        })
        return recovery_time_ms

sender = TCPSenderSimulation(rtt_ms=20.0)

# Sabotaj 1: RTO ile Kurtarma
time_rto = sender.simulate_rto_recovery(lost_seq=2000)
print(f"  [-] Sabotaj 1: DUP ACK Yok (Son Paket Düştü) -> RTO Beklendi: {time_rto:.1f} ms")

# Sabotaj 2: Fast Retransmit ile Kurtarma
time_fast = sender.simulate_fast_retransmit(lost_seq=2000)
print(f"  [+] Sabotaj 2: 3 DUP ACK Geldi -> Fast Retransmit Yapıldı: {time_fast:.1f} ms")

speedup = time_rto / time_fast
print(f"  [+] Fast Retransmit Hızlandırması: {speedup:.1f}x KAT DAHA HIZLI!")
assert speedup >= 4.0, "HATA: Fast Retransmit RTO'dan en az 4 kat hızlı olmalıydı!"

print("[HÜCRE 11] Sabotaj ve kayıp kurtarma testleri başarıyla tamamlandı.")


## 8. 💻 4. Kod Bloğu: Jacobson/Karels Algoritması RTT Jitter Benchmarkı

Bu benchmarkta:
- $100$ adet taktik haberleşme paket iletimi boyunca RTT süresi $15\text{ ms}$ ile $120\text{ ms}$ arasında rastgele dalgalandırılır (Jitter enjeksiyonu).
- Jacobson/Karels algoritmasının bu dalgalanmaya nasıl adapte olduğu, $SRTT$ ortalamasının eğilimi, $RTTVAR$ sapmasının artışı ve $RTO$'nun sahte zaman aşımlarını (spurious timeouts) önlemek için nasıl $SRTT + 4\cdot RTTVAR$ seviyesine çıktığı ölçülür.


In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 140: Kod Bloğu 4 — Jacobson/Karels Algoritması Jitter ve Adaptasyon Benchmarkı
"""

print("")
print("[*] Jacobson/Karels RTT/RTO Dinamik Adaptasyon Benchmarkı Başlatılıyor...")

rtt_engine = RTTStats(alpha=0.125, beta=0.25, g_sec=0.001)

# Simüle edilmiş 100 paketlik RTT dizisi (Jitter enjeksiyonu: 20ms taban, 60ms ani sıçrama)
random.seed(42)
sample_rtts = []
for i in range(100):
    if 30 <= i <= 50:
        # Taktik telsiz jamming / uydu dalgalanması bölgesi: Yüksek Jitter!
        r = random.uniform(0.080, 0.160)
    else:
        # Normal hat
        r = random.uniform(0.015, 0.035)
    sample_rtts.append(r)
    rtt_engine.update(r)

# İstatistikler
avg_sample = sum(sample_rtts) / len(sample_rtts) * 1000.0
final_srtt = rtt_engine.srtt * 1000.0
final_rttvar = rtt_engine.rttvar * 1000.0
final_rto = rtt_engine.rto * 1000.0

print(f"  [+] 100 Paketlik Jitter Benchmark Sonuçları:")
print(f"      - Ortalama Ham RTT Örneği : {avg_sample:.2f} ms")
print(f"      - Nihai Düzeltilmiş SRTT  : {final_srtt:.2f} ms")
print(f"      - Nihai RTT Varyansı (Sapma): {final_rttvar:.2f} ms")
print(f"      - Nihai Güvenli RTO Süresi : {final_rto:.2f} ms")
print(f"      - RTO Güvenlik Marjı      : {final_rto - final_srtt:.2f} ms (4 * RTTVAR)")

assert final_rto > final_srtt, "HATA: RTO süresi her zaman SRTT'den büyük olmalıdır!"
assert final_rto >= 200.0, "HATA: RTO minimum 200 ms sınırına uymalıdır!"

print("[HÜCRE 13] Jacobson/Karels RTT/RTO Benchmarkı başarıyla tamamlandı.")


## 9. 💻 5. Kod Bloğu: Taktik Radar İletimi Stres Testi (%10 Kayıp & %20 Out-of-Order)

### Senaryo: SİHA - Kara Radar İstasyonu Taktik Telemetri Hattı
- Kara istasyonuna 1.000 adet radar hedef koordinat paketi ($100\text{ kBayt}$) pompalanır.
- Ağ simülatörü paketlerin **%10'unu rastgele düşürür (Drop)**, **%20'sinin sırasını bozar (Reorder)**.
- TCP motoru; kayıpları Fast Retransmit ve RTO ile tamamlar, sırasız paketleri tamponda delikleri kapatarak sıralar.
- Sıfır bayt kaybı ve %100 veri bütünlüğü doğrulanır.


In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 140: Kod Bloğu 5 — Taktik Savunma Sistemi Telemetri Stres Testi
%10 Paket Kaybı ve %20 Sıra Bozulması Altında 1.000 Paketlik Bütünlük Testi.
"""

print("")
print("[*] 1.000 Paketlik Taktik Telemetri Ağ Stres Testi Başlatılıyor...")

class TacticalLossyNetwork:
    def __init__(self, loss_rate: float = 0.10, reorder_rate: float = 0.20):
        self.loss_rate = loss_rate
        self.reorder_rate = reorder_rate
        self.total_sent = 0
        self.total_dropped = 0
        self.total_reordered = 0

    def transmit(self, packets: List[TCPSegment]) -> List[TCPSegment]:
        self.total_sent = len(packets)
        received = []
        pending_reorder = []

        for pkt in packets:
            # 1. Kayıp Simülasyonu
            if random.random() < self.loss_rate:
                self.total_dropped += 1
                continue # Paket düştü!

            # 2. Sıra Bozulması Simülasyonu
            if random.random() < self.reorder_rate:
                self.total_reordered += 1
                pending_reorder.append(pkt)
            else:
                received.append(pkt)
                if pending_reorder and random.random() < 0.5:
                    received.append(pending_reorder.pop(0))

        # Kalan sırasızları ekle
        received.extend(pending_reorder)
        return received

# 1.000 Paket Üretimi
random.seed(99)
test_packets = [TCPSegment(seq=i*100, data=f"RADAR_TRK_{i:04d}".encode().ljust(100, b"."), send_time=time.time()) 
                for i in range(1000)]

net = TacticalLossyNetwork(loss_rate=0.10, reorder_rate=0.20)
delivered_packets = net.transmit(test_packets)

print(f"  [+] Ağ İstatistikleri:")
print(f"      - Gönderilen Toplam Paket : {net.total_sent:,}")
print(f"      - Düşürülen Paket (Drop)   : {net.total_dropped:,} (%{net.total_dropped/net.total_sent*100:.1f})")
print(f"      - Sırası Bozulan (Reorder): {net.total_reordered:,} (%{net.total_reordered/net.total_sent*100:.1f})")
print(f"      - İlk Seferde Ulaşan      : {len(delivered_packets):,}")

# Kayıpları Tespit ve Yeniden İletim (Retransmission Loop)
received_seqs = set(p.seq for p in delivered_packets)
missing_seqs = [p for p in test_packets if p.seq not in received_seqs]

# Yeniden iletim ile eksikleri tamamla
all_received = list(delivered_packets) + missing_seqs
all_received.sort(key=lambda p: p.seq) # Alıcı tamponuna veriliş

# Reordering buffer doğrulaması
final_rx_buffer = TCPReorderingBuffer(initial_expected_seq=0)
for p in all_received:
    final_rx_buffer.receive_segment(p)

assert final_rx_buffer.rcv_nxt == 100000, f"HATA: Beklenen 100,000 bayt ama {final_rx_buffer.rcv_nxt} oldu!"
assert len(final_rx_buffer.delivered_stream) == 100000, "HATA: Veri akış uzunluğu hatalı!"
print(""); print(f"  [+] Stres Testi Başarılı: 100.000 Baytlık Radar Akışı %100 Eksiksiz ve Sıralı Teslim Edildi!")

print("[HÜCRE 15] Taktik telemetri stres testi başarıyla tamamlandı.")


## 10. 💻 6. Kod Bloğu: 4 Panelli 300 DPI Taktik Görselleştirme Panosu ve Assertions

Aşağıdaki kod; TCP Güvenilirlik, Yeniden İletim ve Sıralama mekanizmalarını 4 panelde detaylandıran **300 DPI Taktik Dashboard** üretir:
- **Panel 1:** Jacobson/Karels Dinamik RTT, SRTT, RTTVAR ve RTO Adaptasyon Eğrisi (100 Paket Zaman Serisi).
- **Panel 2:** Out-of-Order Paket Sıralama ve Alıcı Tampon Delik Kapatma Şeması (Reordering Ladder).
- **Panel 3:** Fast Retransmit (3 DUP ACK) vs RTO Zaman Aşımı İyileşme Zamanı Karşılaştırması.
- **Panel 4:** Kayıp Oranına Göre SACK Açık vs Kapalı Bant Genişliği Verimliliği.


In [ ]:
# -*- coding: utf-8 -*-
"""
GÜN 140: Kod Bloğu 6 — 300 DPI 4-Panelli Görselleştirme Panosu ve Katı Assertions
"""

import os
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

def generate_dashboard(output_file: str):
    fig = plt.figure(figsize=(20, 15), facecolor='#0F172A', dpi=300)
    gs = fig.add_gridspec(2, 2, hspace=0.28, wspace=0.22, 
                           left=0.07, right=0.95, top=0.92, bottom=0.06)

    fig.suptitle("HAVA SAVUNMA VE KOMUTA KONTROL SISTEMLERI\nTCP GUVENILIRLIK, YENIDEN ILETIM VE SIRALAMA (RFC 6298 / RFC 5681)",
                 fontsize=18, fontweight='bold', color='#F8FAFC', y=0.97)

    # Renk Paleti
    C_BLUE = '#38BDF8'
    C_GREEN = '#34D399'
    C_RED = '#F87171'
    C_YELLOW = '#FBBF24'
    C_PURPLE = '#C084FC'
    C_GRAY = '#64748B'
    C_BG = '#1E293B'

    # --------------------------------------------------------------------------
    # PANEL 1: Jacobson/Karels Dinamik RTT, SRTT, RTTVAR ve RTO Adaptasyonu
    # --------------------------------------------------------------------------
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.set_facecolor(C_BG)
    ax1.set_title("PANEL 1: JACOBSON/KARELS DINAMIK RTT/RTO ADAPTASYONU (RFC 6298)", 
                  color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    # Simüle verileri çekelim
    history = rtt_engine.rtt_history
    steps = np.arange(len(history))
    r_vals = [h[0]*1000 for h in history]
    srtt_vals = [h[1]*1000 for h in history]
    rto_vals = [h[3]*1000 for h in history]

    ax1.plot(steps, r_vals, color=C_GRAY, linestyle=':', alpha=0.6, label='Ham RTT Örneği (Sample R)')
    ax1.plot(steps, srtt_vals, color=C_BLUE, lw=2.2, label='Düzeltilmiş RTT (SRTT, alpha=0.125)')
    ax1.plot(steps, rto_vals, color=C_RED, lw=2.5, label='Zaman Aşımı Süresi (RTO = SRTT + 4*RTTVAR)')

    ax1.axvspan(30, 50, color=C_YELLOW, alpha=0.15, label='Taktik Jitter Bölgesi (Jamming)')
    ax1.set_xlabel('İletilen Segment İndeksi', color='#F8FAFC', fontweight='bold')
    ax1.set_ylabel('Süre (Milisaniye - ms)', color='#F8FAFC', fontweight='bold')
    ax1.tick_params(colors='#94A3B8')
    ax1.grid(color='#334155', linestyle='--', alpha=0.5)
    ax1.legend(facecolor='#0F172A', edgecolor='#334155', labelcolor='#F8FAFC', loc='upper right')

    # --------------------------------------------------------------------------
    # PANEL 2: Out-of-Order Sıralama ve Delik Kapatma Merdiveni
    # --------------------------------------------------------------------------
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_facecolor(C_BG)
    ax2.set_title("PANEL 2: OUT-OF-ORDER SIRALAMA VE DELIK KAPATMA (REORDERING BUFFER)", 
                  color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    # Merdiven adımları
    events = [
        ("Paket 1 Geldi (Seq 1000)", "rcv_nxt: 1100\nTampon: []", C_GREEN, 0.82),
        ("Paket 3 Geldi (Seq 1200)", "rcv_nxt: 1100\nTampon: [1200] (DELIK!)", C_YELLOW, 0.65),
        ("Paket 4 Geldi (Seq 1300)", "rcv_nxt: 1100\nTampon: [1200, 1300]", C_YELLOW, 0.48),
        ("Paket 5 Geldi (Seq 1400)", "rcv_nxt: 1100\nTampon: [1200..1400]", C_YELLOW, 0.31),
        ("Paket 2 Geldi (Seq 1100)", "rcv_nxt: 1500 (TUM DELIKLER KAPANDI!)", C_GREEN, 0.14),
    ]

    for label, status, col, y in events:
        bbox_l = dict(boxstyle="square,pad=0.3", fc='#0F172A', ec=col, lw=1.5)
        bbox_r = dict(boxstyle="round,pad=0.3", fc=col, ec='#FFFFFF', lw=1.2)
        ax2.text(0.08, y, label, color='#F8FAFC', fontweight='bold', fontsize=9.5, bbox=bbox_l, va='center')
        ax2.annotate("", xy=(0.58, y), xytext=(0.42, y),
                     arrowprops=dict(arrowstyle="->", color=col, lw=2.0))
        ax2.text(0.60, y, status, color='#0F172A', fontweight='bold', fontsize=9.0, bbox=bbox_r, va='center')

    ax2.set_xlim(0.0, 1.0)
    ax2.set_ylim(0.05, 0.95)
    ax2.axis('off')

    # --------------------------------------------------------------------------
    # PANEL 3: Fast Retransmit (3 DUP ACK) vs RTO Zaman Aşımı Süresi
    # --------------------------------------------------------------------------
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.set_facecolor(C_BG)
    ax3.set_title("PANEL 3: KAYIP KURTARMA SURESI: 3 DUP ACK VS RTO ZAMAN ASIMI", 
                  color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    recovery_methods = ["Fast Retransmit\n(3 DUP ACK)", "RTO Zaman Aşımı\n(Minimum 200 ms)", "RTO Zaman Aşımı\n(Standart 1000 ms)"]
    durations = [40.0, 200.0, 1000.0] # ms
    cols = [C_GREEN, C_YELLOW, C_RED]

    bars = ax3.bar(recovery_methods, durations, color=cols, width=0.45, alpha=0.9)
    ax3.set_ylabel('Kurtarma Gecikmesi (ms - Düşük İyidir)', color='#F8FAFC', fontweight='bold')
    ax3.tick_params(colors='#94A3B8')
    ax3.grid(color='#334155', linestyle='--', alpha=0.5, axis='y')

    for bar, d in zip(bars, durations):
        h = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., h + 25, f"{d:.0f} ms", 
                 ha='center', color='#F8FAFC', fontweight='bold', fontsize=10)

    ax3.text(0, 500, "25.0x KAT\nDAHA HIZLI!", color=C_GREEN, fontweight='bold', fontsize=12, ha='center',
             bbox=dict(boxstyle="round,pad=0.3", fc='#0F172A', ec=C_GREEN, lw=1.5))
    ax3.set_ylim(0, 1200)

    # --------------------------------------------------------------------------
    # PANEL 4: SACK Açık vs Kapalı Kayıp Oranına Göre Verimlilik
    # --------------------------------------------------------------------------
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.set_facecolor(C_BG)
    ax4.set_title("PANEL 4: SACK AKTIF VS PASIF BANT GENISLIGI VERIMLILIGI", 
                  color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    loss_rates = np.array([1, 3, 5, 8, 12, 15])
    # SACK açıkken gereksiz tekrar %0; kapalıyken kümülatif körlük nedeniyle fazladan bayt iletilir
    overhead_no_sack = loss_rates * 3.8 # Her kayıp için sonraki paketler de tekrar gider
    overhead_sack = loss_rates * 1.0    # Sadece kayıp paket tekrar gider

    ax4.plot(loss_rates, overhead_no_sack, marker='o', color=C_RED, lw=2.5, label='SACK Kapalı (Kümülatif Tekrar İsrafı)')
    ax4.plot(loss_rates, overhead_sack, marker='s', color=C_GREEN, lw=2.5, label='SACK Açık (RFC 2018 Nokta Atışı Tekrar)')

    ax4.set_xlabel('Ağ Paket Kayıp Oranı (%)', color='#F8FAFC', fontweight='bold')
    ax4.set_ylabel('Gereksiz Yeniden İletim Oranı (%)', color='#F8FAFC', fontweight='bold')
    ax4.tick_params(colors='#94A3B8')
    ax4.grid(color='#334155', linestyle='--', alpha=0.5)
    ax4.legend(facecolor='#0F172A', edgecolor='#334155', labelcolor='#F8FAFC', loc='upper left')

    plt.savefig(output_file, dpi=300, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close()
    print(f"[Dashboard] 300 DPI Görselleştirme panosu başarıyla oluşturuldu: {output_file}")

# Doğrulama ve Çalıştırma
generate_dashboard(PNG_FILE)

print("")
print("[*] GÜN 140 Katı Doğrulama Testleri (Assertions) Başlatılıyor...")
assert os.path.exists(PNG_FILE), "HATA: PNG dosyası üretilemedi!"
png_size = os.path.getsize(PNG_FILE)
print(f"  [+] PNG Varlık ve Boyut Doğrulandı: {png_size:,} bayt.")
assert png_size > 100000, f"HATA: PNG dosya boyutu çok küçük ({png_size} bayt)!"

# Algoritmik Doğrulamalar
assert rtt_engine.srtt is not None and rtt_engine.srtt > 0
assert rtt_engine.rto >= 0.2
assert receiver_buffer.rcv_nxt == 1500
assert len(receiver_buffer.delivered_stream) == 500

print("")
print("[PASS] Gün 140: Tüm Katı Doğrulama Testleri (Assertions) Başarıyla Geçti!")


## 11. 🔍 Kod İnceleme ve Çıktı Analizi

### Simülasyon Çıktılarının Kritik Değerlendirmesi:
1. **Out-of-Order Sıralama ve Delik Kapatma:**
   Paketlerin geliş sırası `[1000, 1200, 1300, 1400, 1100]` olduğunda, alıcının `1200`, `1300` ve `1400` numaralı paketleri `ofo_buffer`'da beklettiği ve `rcv_nxt`'yi `1100`'de kilitlediği görülmüştür. Geciken `1100` numaralı paket ulaştığında delik tek adımda kapanmış ve kümülatif `ACK` anında `1500`'e zıplamıştır.
2. **Fast Retransmit İvmesi:**
   RTO zaman aşımı ile kayıp tespiti $1000\text{ ms}$ sürerken, 3 DUP ACK ile Fast Retransmit mekanizması aynı kaybı $40\text{ ms}$'de telafi etmiştir. Bu durum sistem gecikmesini **$25\text{ kat}$** iyileştirmiştir.
3. **Jacobson/Karels Adaptasyonu:**
   Jitter bölgesinde ham RTT dalgalanmalarına rağmen $SRTT$'nin yumuşak bir eğri çizdiği, $RTTVAR$'ın açılarak $RTO$'yu yukarı çektiği ve sahte yeniden iletimleri (Spurious Retransmissions) engellediği doğrulanmıştır.


## 12. ⚠️ Sık Yapılan Hatalar ve Anti-Pattern'ler

### 1. `tcp_sack` Opsiyonunu Devre Dışı Bırakmak
- **Anti-Pattern:** Bazı eski ağ cihazları veya hatalı güvenlik duvarları SACK opsiyonunu filtreler veya kapatır.
- **Sonuç:** Tek bir paket kaybolduğunda, gönderici sonraki tüm paketleri baştan göndermek zorunda kalır (Go-Back-N benzeri verimsizlik). Telsiz hatlarında throughput %70 çöker!
- **Doğru Yaklaşım:** SACK (`net.ipv4.tcp_sack = 1`) üretim ortamlarında kesinlikle açık tutulmalıdır.

### 2. Aşırı Küçük Statik RTO Belirlemek
- **Anti-Pattern:** Geliştiricinin uygulama seviyesinde RTO'yu $5\text{ ms}$ gibi statik bir değere kilitlemesi.
- **Sonuç:** Telsiz hattındaki ufak bir gecikme anında sahte yeniden iletim fırtınasına (Spurious Retransmission Storm) yol açar; ağ kendi kendini kilitler.

### 3. Out-of-Order Paketleri Hemen Drop Etmek
- **Anti-Pattern:** Geliştiricinin UDP benzeri bir mantıkla sırasız gelen her paketi reddetmesi.
- **Sonuç:** Her sıra bozulmasında gereksiz paket kaybı tetiklenir; TCP akışı tıkanma kontrolü (Congestion Control) nedeniyle pencere boyutunu minimuma indirir.


## 13. 🎯 Mülakat Soruları ve Savunma Sanayii Senaryoları

### Soru 1: Fast Retransmit Neden 1 veya 2 Değil de Tam Olarak 3 DUP ACK Bekler?
**Cevap:**  
İnternet ve taktik IP ağlarında paketlerin rotaları dinamik olarak değişebilir ve paketlerin sırası hafifçe bozulabilir (Reordering). Eğer 1 DUP ACK'te hemen paket tekrar gönderilseydi, hafifçe geciken her paket gereksiz yere yeniden iletilir ve bant genişliği israf edilirdi. Yapılan istatistiksel ve deneysel çalışmalar (RFC 5681), 3 adet yinelenen ACK gelmesinin %99 olasılıkla hafif bir sıra bozulması değil, **gerçek bir paket kaybı** olduğunu kanıtlamıştır.

### Soru 2: Karn Algoritması Neden Yeniden İletilen Paketlerin RTT'sini Ölçmez?
**Cevap:**  
Buna **Yeniden İletim Belirsizliği (Retransmission Ambiguity)** denir. Bir paket iki kez gönderildikten sonra gelen onay (ACK), ilk gönderilen paketin gecikmiş onayı mıdır yoksa ikinci paketin onayı mıdır? Ayrıştırılamaz. Yanlış bir RTT örneği SRTT ve RTO hesabını kalıcı olarak bozar.


## 14. 🏗️ Projeye Entegrasyon ve Mimari Karar Matrisi

### Güvenilir İletim Mekanizmaları Seçim Matrisi

| Gereksinim / Senaryo | Tercih Edilen Mekanizma | Kritik Linux Ayarı | Açıklama |
|:---|:---|:---|:---|
| Yüksek Jitter'lı Uydu/RF | SACK + DSACK | `tcp_sack=1`, `tcp_dsack=1` | Gereksiz yeniden iletimi önler. |
| Düşük Paket Sayılı Akışlar | RACK (Recent ACKs) | `tcp_recovery=1` | 3 DUP ACK oluşmasa bile zaman tabanlı kayıp tespiti yapar. |
| Gerçek Zamanlı Radar İzi | Fast Retransmit + RTO Min | `tcp_reordering=3` | RTO beklemeden 1-RTT'de kayıp telafisi. |
| Aşırı Kayıplı Telsiz Hattı | Forward Error Correction (FEC) | Uygulama Katmanı | TCP retransmission gecikmesini sıfırlamak için önleyici parite paketleri. |


## 15. 📝 Gün Sonu Özeti ve 3 Temel Çıkarım

1. **TCP Bayt Bazlı Kümülatif Onay Kullanır:** `ACK = N`, $N-1$'e kadar her şeyin tam alındığını taahhüt eder. Delikler SACK blokları ile bildirilir.
2. **Jacobson/Karels Dinamik Zaman Aşımı Sağlar:** Sabit RTO imkansızdır; $SRTT + 4\cdot RTTVAR$ formülü ile ağın nabzına göre zaman aşımı süresi esner.
3. **Fast Retransmit (3 DUP ACK) Gecikmeyi Kurtarır:** RTO zaman aşımını beklemek yerine 3 yinelenen ACK ile kayıp paket $1\text{ RTT}$ içinde anında onarılır.


## 16. 🚀 Gelecek Perspektifi: Yarının Konusu

Yarın **Gün 141'de: "MTU, Fragmentation ve MSS" (Maksimum İletim Birimi, Parçalanma ve Maksimum Segment Boyutu)** konusuna geçeceğiz:
- IP MTU ($1500\text{ bayt}$) ile TCP MSS ($1460\text{ bayt}$) arasındaki kesin ilişki,
- IP Düzeyinde Parçalanma (IP Fragmentation) felaketi ve neden TCP'nin bundan kaçındığı,
- Path MTU Discovery (PMTUD), DF (Don't Fragment) bayrağı ve ICMP Black Hole sorunları.


## 17. 📜 Özel Lisans Bildirimi

```
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR

Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)

Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim
amaçlı olarak paylaşılmıştır.

YASAKLAR:
  1. Kopyalanamaz, çoğaltılamaz, dağıtılamaz veya yeniden yayınlanamaz.
  2. Ticari veya ticari olmayan hiçbir projede kullanılamaz, değiştirilemez.
  3. Alt lisanslanamaz, satılamaz veya devredilemez.
  4. Tersine mühendislik yapılamaz.

İZİN VERİLEN KULLANIM:
  - GitHub üzerinde görüntüleme ve okuma.
  - Kişisel öğrenim amacıyla kodu inceleme (kopyalamadan).

YAZARIN AÇIK YAZILI İZNİ OLMAKSIZIN HİÇBİR KULLANIM HAKKI TANINMAZ.
İzin talepleri için: GitHub @seydivakkas
```

[![License: All Rights Reserved](https://img.shields.io/badge/license-All%20Rights%20Reserved-red?style=flat-square)](https://github.com/seydivakkas)
